# Dataclasses in Python

`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__` from class fields.

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(p)          # Point(x=1.0, y=2.0)
print(p.x, p.y)   # 1.0 2.0

## `__post_init__`

Called **automatically** by the generated `__init__` after all fields are set.

Use it for:
- Validation
- Derived fields
- Any logic that needs all fields to exist first

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

    def __post_init__(self):
        if self.price < 0:
            raise ValueError(f"Price cannot be negative: {self.price}")
        self.name = self.name.strip().title()  # normalise name

p = Product(name="  apple  ", price=1.5)
print(p)   # Product(name='Apple', price=1.5)

try:
    Product(name="bad", price=-1)
except ValueError as e:
    print(e)

## Is `__post_init__` only for dataclasses?

Yes — it is a **dataclass-specific hook**.

The generated `__init__` explicitly calls `self.__post_init__()` at the end.

In a plain class, `__post_init__` has no special meaning and will never be called automatically.

## `__new__` — object creation step

Python object lifecycle:

```
ClassName(args)
      ↓
__new__(cls, *args, **kwargs)   ← creates the instance
      ↓
__init__(self, *args, **kwargs) ← initialises the instance
```

`__new__` is a **static method** (implicitly). Its first argument is always `cls` (the class), not `self`.

It **must return** an instance; that instance is then passed to `__init__` as `self`.

In [ ]:
class Discount(float):
    """A float subclass that enforces 0 <= value <= 100."""

    def __new__(cls, value):
        val = float(value)
        if not (0 <= val <= 100):
            raise ValueError(f"Discount must be 0-100, got {val}")
        return super().__new__(cls, val)  # super().__new__ creates the float

d = Discount(20)
print(d, type(d))   # 20.0 <class '__main__.Discount'>

try:
    Discount(150)
except ValueError as e:
    print(e)

### Why use `__new__` instead of `__init__`?

- **Immutable types** (`int`, `float`, `str`, `tuple`) must use `__new__` because the value is set at creation time — `__init__` is too late.
- Singleton patterns.
- Custom metaclass logic.

## `__new__` inside a dataclass

`__new__` and `__post_init__` serve different purposes and can coexist:

| Hook | Runs | Purpose |
|---|---|---|
| `__new__` | Before `__init__` | Allocates / returns the instance |
| `__init__` | Auto-generated by `@dataclass` | Sets field values |
| `__post_init__` | End of `__init__` | Validates / transforms after fields are set |

In [ ]:
from dataclasses import dataclass

@dataclass
class Temperature:
    celsius: float

    def __post_init__(self):
        if self.celsius < -273.15:
            raise ValueError("Below absolute zero")
        self.fahrenheit = self.celsius * 9 / 5 + 32  # derived field

t = Temperature(100)
print(t.celsius, t.fahrenheit)   # 100 212.0